In [1]:

from pathlib import Path
import json
import warnings

import numpy as np
import pandas as pd
import joblib

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

warnings.filterwarnings("ignore")

# Project root: assumes this notebook is inside notebooks/
PROJECT_ROOT = Path.cwd().parent

DATA_DIR = PROJECT_ROOT / "data"
PROCESSED_DIR = DATA_DIR / "processed"
MODELS_DIR = PROJECT_ROOT / "models"
REPORT_DIR = PROJECT_ROOT / "reports" / "model_results"

REPORT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
FORECAST_HORIZON = 7

print("Project root:", PROJECT_ROOT)
print("Processed data directory:", PROCESSED_DIR)
print("Models directory:", MODELS_DIR)
print("Forecast horizon:", FORECAST_HORIZON, "days")


Project root: D:\AI-Retail-Demand-Forecasting
Processed data directory: D:\AI-Retail-Demand-Forecasting\data\processed
Models directory: D:\AI-Retail-Demand-Forecasting\models
Forecast horizon: 7 days


In [2]:

candidate_files = [
    PROCESSED_DIR / "cleaned_sales_data.csv",
    PROCESSED_DIR / "model_dataset.csv",
    PROCESSED_DIR / "train.csv",
    PROCESSED_DIR / "validation.csv",
    PROCESSED_DIR / "test.csv",
    MODELS_DIR / "best_demand_model.pkl",
    MODELS_DIR / "model_metadata.json",
]

for path in candidate_files:
    status = "FOUND" if path.exists() else "NOT FOUND"
    print(f"{status}: {path.relative_to(PROJECT_ROOT)}")


FOUND: data\processed\cleaned_sales_data.csv
FOUND: data\processed\model_dataset.csv
FOUND: data\processed\train.csv
FOUND: data\processed\validation.csv
FOUND: data\processed\test.csv
FOUND: models\best_demand_model.pkl
FOUND: models\model_metadata.json


In [3]:

# Load processed datasets
cleaned_df = pd.read_csv(PROCESSED_DIR / "cleaned_sales_data.csv")
model_df = pd.read_csv(PROCESSED_DIR / "model_dataset.csv")

train_df = pd.read_csv(PROCESSED_DIR / "train.csv")
validation_df = pd.read_csv(PROCESSED_DIR / "validation.csv")
test_df = pd.read_csv(PROCESSED_DIR / "test.csv")

# Load the previously saved model and its metadata
model_path = MODELS_DIR / "best_demand_model.pkl"
metadata_path = MODELS_DIR / "model_metadata.json"

model = joblib.load(model_path)

with open(metadata_path, "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("DATASET SHAPES")
print("-" * 45)
print("Cleaned data:", cleaned_df.shape)
print("Model dataset:", model_df.shape)
print("Train:", train_df.shape)
print("Validation:", validation_df.shape)
print("Test:", test_df.shape)

print("\nMODEL TYPE")
print("-" * 45)
print(type(model))

print("\nMODEL METADATA")
print("-" * 45)
print(json.dumps(metadata, indent=2))


DATASET SHAPES
---------------------------------------------
Cleaned data: (76000, 16)
Model dataset: (73000, 55)
Train: (51000, 55)
Validation: (11000, 55)
Test: (11000, 55)

MODEL TYPE
---------------------------------------------
<class 'sklearn.pipeline.Pipeline'>

MODEL METADATA
---------------------------------------------
{
  "target": "Demand",
  "model": "XGBoost",
  "best_parameters": {
    "n_estimators": 500,
    "max_depth": 7,
    "learning_rate": 0.08
  },
  "number_of_features": 53,
  "categorical_features": [
    "Store ID",
    "Product ID",
    "Category",
    "Region",
    "Weather Condition",
    "Promotion",
    "Seasonality",
    "Epidemic"
  ],
  "numerical_features": [
    "Year",
    "Month",
    "Quarter",
    "DayOfMonth",
    "DayOfWeek",
    "WeekOfYear",
    "IsWeekend",
    "DayOfWeek_Sin",
    "DayOfWeek_Cos",
    "Month_Sin",
    "Month_Cos",
    "Lag_1",
    "Lag_7",
    "Lag_14",
    "Lag_30",
    "RollingMean_7",
    "RollingMean_14",
    "RollingMe

In [4]:

print("CLEANED DATA COLUMNS")
print(cleaned_df.columns.tolist())

print("\nMODEL DATASET COLUMNS")
print(model_df.columns.tolist())

print("\nTRAINING DATA COLUMNS")
print(train_df.columns.tolist())

print("\nTEST DATA COLUMNS")
print(test_df.columns.tolist())

# Inspect feature names stored by the fitted estimator, if available
estimator = model

if hasattr(estimator, "named_steps"):
    print("\nPIPELINE STEPS:")
    print(list(estimator.named_steps.keys()))

    final_estimator = list(estimator.named_steps.values())[-1]
else:
    final_estimator = estimator

if hasattr(final_estimator, "feature_names_in_"):
    print("\nFEATURE NAMES KNOWN TO FINAL ESTIMATOR:")
    print(final_estimator.feature_names_in_.tolist())
elif hasattr(model, "feature_names_in_"):
    print("\nFEATURE NAMES KNOWN TO MODEL:")
    print(model.feature_names_in_.tolist())
else:
    print("\nFeature names are not directly exposed by the loaded estimator.")
    print("We will use the metadata and pipeline structure to identify them.")


CLEANED DATA COLUMNS
['Date', 'Store ID', 'Product ID', 'Category', 'Region', 'Inventory Level', 'Units Sold', 'Units Ordered', 'Price', 'Discount', 'Weather Condition', 'Promotion', 'Competitor Pricing', 'Seasonality', 'Epidemic', 'Demand']

MODEL DATASET COLUMNS
['Date', 'Year', 'Month', 'Quarter', 'DayOfMonth', 'DayOfWeek', 'WeekOfYear', 'IsWeekend', 'DayOfWeek_Sin', 'DayOfWeek_Cos', 'Month_Sin', 'Month_Cos', 'Lag_1', 'Lag_7', 'Lag_14', 'Lag_30', 'RollingMean_7', 'RollingMean_14', 'RollingMean_30', 'RollingStd_7', 'RollingStd_14', 'RollingStd_30', 'RollingMin_7', 'RollingMax_7', 'RollingMin_30', 'RollingMax_30', 'Inventory_Lag_1', 'Inventory_Lag_7', 'UnitsSold_Lag_1', 'UnitsSold_Lag_7', 'UnitsSold_Lag_14', 'UnitsOrdered_Lag_1', 'UnitsOrdered_Lag_7', 'UnitsOrdered_Lag_14', 'Inventory Level', 'Price', 'Discount', 'Competitor Pricing', 'Price_Lag_1', 'Price_Change', 'Price_Change_Pct', 'CompetitorPrice_Lag_1', 'CompetitorPrice_Change', 'Price_Competitor_Difference', 'Inventory_Demand_R

In [5]:

DATE_COL = "Date"
STORE_COL = "Store ID"
PRODUCT_COL = "Product ID"
TARGET_COL = "Demand"

cleaned_df[DATE_COL] = pd.to_datetime(
    cleaned_df[DATE_COL], errors="coerce"
)

print("Date range:")
print(cleaned_df[DATE_COL].min(), "to", cleaned_df[DATE_COL].max())

print("\nMissing dates in Date column:")
print(cleaned_df[DATE_COL].isna().sum())

print("\nDuplicate store-product-date rows:")
duplicate_count = cleaned_df.duplicated(
    subset=[STORE_COL, PRODUCT_COL, DATE_COL]
).sum()
print(duplicate_count)

# Check the spacing between consecutive records for each series
series_df = cleaned_df.sort_values(
    [STORE_COL, PRODUCT_COL, DATE_COL]
).copy()

series_df["Days_Between_Records"] = (
    series_df.groupby([STORE_COL, PRODUCT_COL])[DATE_COL]
    .diff()
    .dt.days
)

print("\nDays between consecutive observations:")
print(series_df["Days_Between_Records"].value_counts(dropna=False).sort_index())

print("\nSeries with gaps larger than one day:")
gap_counts = (
    series_df.loc[series_df["Days_Between_Records"] > 1]
    .groupby([STORE_COL, PRODUCT_COL])
    .size()
)

print("Number of affected store-product combinations:", len(gap_counts))


Date range:
2022-01-01 00:00:00 to 2024-01-30 00:00:00

Missing dates in Date column:
0

Duplicate store-product-date rows:
0

Days between consecutive observations:
Days_Between_Records
1.0    75900
NaN      100
Name: count, dtype: int64

Series with gaps larger than one day:
Number of affected store-product combinations: 0


In [6]:

# Check whether Demand and Units Sold contain overlapping information
print("DEMAND VS UNITS SOLD")
print("-" * 45)

print(cleaned_df[["Demand", "Units Sold"]].describe())

print(
    "\nExact-match percentage:",
    round(
        cleaned_df["Demand"].eq(cleaned_df["Units Sold"]).mean() * 100,
        2
    ),
    "%"
)

print(
    "\nCorrelation:",
    cleaned_df[["Demand", "Units Sold"]].corr().iloc[0, 1]
)

# Inspect feature values at the end of the available history
latest_date = cleaned_df["Date"].max()
latest_rows = (
    cleaned_df[cleaned_df["Date"] == latest_date]
    .sort_values(["Store ID", "Product ID"])
)

print("\nLATEST DATE:", latest_date)
print("Rows on latest date:", len(latest_rows))

cols_to_inspect = [
    "Store ID",
    "Product ID",
    "Inventory Level",
    "Units Sold",
    "Units Ordered",
    "Price",
    "Discount",
    "Promotion",
    "Weather Condition",
    "Seasonality",
    "Epidemic",
    "Demand"
]

print("\nLATEST OBSERVATIONS")
display(latest_rows[cols_to_inspect].head(10))

# Check whether promotion, discount, and price vary over time
print("\nNUMBER OF DISTINCT VALUES IN THE DATA")
for col in [
    "Price",
    "Discount",
    "Promotion",
    "Weather Condition",
    "Seasonality",
    "Epidemic"
]:
    print(f"{col}: {cleaned_df[col].nunique()}")


DEMAND VS UNITS SOLD
---------------------------------------------
             Demand    Units Sold
count  76000.000000  76000.000000
mean     104.317158     88.827316
std       46.964801     43.994525
min        4.000000      0.000000
25%       71.000000     58.000000
50%      100.000000     84.000000
75%      133.000000    114.000000
max      430.000000    426.000000

Exact-match percentage: 2.05 %

Correlation: 0.8334214303439825

LATEST DATE: 2024-01-30 00:00:00
Rows on latest date: 100

LATEST OBSERVATIONS


,Store ID,Product ID,Inventory Level,Units Sold,Units Ordered,Price,Discount,Promotion,Weather Condition,Seasonality,Epidemic,Demand
759,S001,P0001,383,68,0,71.56,10,0,Snowy,Winter,0,111
1519,S001,P0002,136,77,117,70.55,0,0,Snowy,Winter,0,119
2279,S001,P0003,434,151,0,59.46,10,0,Snowy,Winter,0,137
3039,S001,P0004,241,86,0,93.82,10,0,Snowy,Winter,0,84
3799,S001,P0005,946,29,0,45.82,0,0,Snowy,Winter,0,29
4559,S001,P0006,436,81,0,37.13,10,0,Snowy,Winter,0,96
5319,S001,P0007,476,64,0,24.57,0,0,Snowy,Winter,0,80
6079,S001,P0008,162,34,0,34.68,10,0,Snowy,Winter,0,53
6839,S001,P0009,128,90,70,90.35,0,0,Snowy,Winter,0,84
7599,S001,P0010,518,64,0,125.82,5,0,Snowy,Winter,0,69



NUMBER OF DISTINCT VALUES IN THE DATA
Price: 15396
Discount: 6
Promotion: 2
Weather Condition: 4
Seasonality: 4
Epidemic: 2


In [7]:

# Feature groups used by the existing fitted pipeline
CATEGORICAL_FEATURES = metadata["categorical_features"]
NUMERICAL_FEATURES = metadata["numerical_features"]

FEATURE_COLUMNS = CATEGORICAL_FEATURES + NUMERICAL_FEATURES

# Confirm the feature count and required columns
assert len(FEATURE_COLUMNS) == metadata["number_of_features"], (
    f"Expected {metadata['number_of_features']} features, "
    f"but metadata lists {len(FEATURE_COLUMNS)}."
)

missing_features = set(FEATURE_COLUMNS) - set(model_df.columns)
assert not missing_features, f"Missing model features: {missing_features}"

assert TARGET_COL in cleaned_df.columns
assert DATE_COL in cleaned_df.columns

print("Feature count:", len(FEATURE_COLUMNS))
print("All required features are present.")
print("Forecast assumption: carry forward latest observed operational values.")


Feature count: 53
All required features are present.
Forecast assumption: carry forward latest observed operational values.


In [8]:

def build_calendar_features(date):
    """Create calendar features for a forecast date."""
    date = pd.Timestamp(date)
    day_of_year = date.dayofyear

    return {
        "Year": date.year,
        "Month": date.month,
        "Quarter": date.quarter,
        "DayOfMonth": date.day,
        "DayOfWeek": date.dayofweek,
        "WeekOfYear": int(date.isocalendar().week),
        "IsWeekend": int(date.dayofweek >= 5),
        "DayOfWeek_Sin": np.sin(2 * np.pi * date.dayofweek / 7),
        "DayOfWeek_Cos": np.cos(2 * np.pi * date.dayofweek / 7),
        "Month_Sin": np.sin(2 * np.pi * date.month / 12),
        "Month_Cos": np.cos(2 * np.pi * date.month / 12),
    }


def recursive_forecast_one_series(history, horizon=7):
    """
    Forecast one Store ID + Product ID series recursively.

    Uses:
    - historical Demand for dates up to the forecast origin
    - predicted Demand for subsequent forecast days
    - last observed values for operational inputs not known in advance

    Returns one row per future day.
    """
    history = history.sort_values("Date").copy().reset_index(drop=True)

    if history.empty:
        raise ValueError("History is empty.")

    if history["Date"].duplicated().any():
        raise ValueError("Duplicate dates found in this series.")

    if history["Date"].diff().dropna().dt.days.ne(1).any():
        raise ValueError("Series does not have continuous daily observations.")

    # Demand history available at the forecast origin
    demand_history = history["Demand"].astype(float).tolist()

    # Last known values for non-target operational variables
    last = history.iloc[-1]

    if len(demand_history) < 30:
        raise ValueError("At least 30 days of history are required.")

    results = []

    for step in range(1, horizon + 1):
        forecast_date = pd.Timestamp(last["Date"]) + pd.Timedelta(days=step)

        row = {
            "Store ID": last["Store ID"],
            "Product ID": last["Product ID"],
            "Category": last["Category"],
            "Region": last["Region"],
            "Weather Condition": last["Weather Condition"],
            "Promotion": last["Promotion"],
            "Seasonality": last["Seasonality"],
            "Epidemic": last["Epidemic"],
            "Inventory Level": last["Inventory Level"],
            "Price": last["Price"],
            "Discount": last["Discount"],
            "Competitor Pricing": last["Competitor Pricing"],
        }

        row.update(build_calendar_features(forecast_date))

        # Lag features from observed history + predictions from earlier steps
        for lag in [1, 7, 14, 30]:
            row[f"Lag_{lag}"] = demand_history[-lag]

        for window in [7, 14, 30]:
            values = np.asarray(demand_history[-window:], dtype=float)
            row[f"RollingMean_{window}"] = values.mean()
            row[f"RollingStd_{window}"] = values.std(ddof=1)

        row["RollingMin_7"] = np.min(demand_history[-7:])
        row["RollingMax_7"] = np.max(demand_history[-7:])
        row["RollingMin_30"] = np.min(demand_history[-30:])
        row["RollingMax_30"] = np.max(demand_history[-30:])

        # Historical operational features: carry forward last known values
        for col in [
            "Inventory Level",
            "Units Sold",
            "Units Ordered",
            "Price",
            "Competitor Pricing",
        ]:
            if col in history.columns:
                values = history[col].astype(float).tolist()
                for lag in [1, 7]:
                    if col == "Inventory Level" and lag in [1, 7]:
                        row[f"Inventory_Lag_{lag}"] = values[-lag]
                    elif col == "Units Sold" and lag in [1, 7]:
                        row[f"UnitsSold_Lag_{lag}"] = values[-lag]
                    elif col == "Units Ordered" and lag in [1, 7]:
                        row[f"UnitsOrdered_Lag_{lag}"] = values[-lag]

                if col == "Units Sold":
                    row["UnitsSold_Lag_14"] = values[-14]
                elif col == "Units Ordered":
                    row["UnitsOrdered_Lag_14"] = values[-14]

        row["Price_Lag_1"] = float(last["Price"])
        row["Price_Change"] = 0.0
        row["Price_Change_Pct"] = 0.0
        row["CompetitorPrice_Lag_1"] = float(last["Competitor Pricing"])
        row["CompetitorPrice_Change"] = 0.0
        row["Price_Competitor_Difference"] = (
            float(last["Price"]) - float(last["Competitor Pricing"])
        )

        row["Inventory_Demand_Ratio"] = (
            float(last["Inventory Level"]) / max(demand_history[-1], 1.0)
        )
        row["Inventory_Coverage_Approx"] = (
            float(last["Inventory Level"])
            / max(np.mean(demand_history[-7:]), 1.0)
        )

        # Ensure the saved model receives exactly the expected feature columns
        X_future = pd.DataFrame([row])[FEATURE_COLUMNS]

        prediction = float(model.predict(X_future)[0])
        prediction = max(0.0, prediction)

        results.append({
            "Date": forecast_date,
            "Store ID": last["Store ID"],
            "Product ID": last["Product ID"],
            "Forecast_Horizon": step,
            "Predicted_Demand": prediction,
        })

        # Crucial: append prediction, not future actual demand
        demand_history.append(prediction)

    return pd.DataFrame(results)


In [9]:

cleaned_for_forecast = cleaned_df.copy()
cleaned_for_forecast["Date"] = pd.to_datetime(cleaned_for_forecast["Date"])

sample_store = cleaned_for_forecast["Store ID"].iloc[0]
sample_product = cleaned_for_forecast["Product ID"].iloc[0]

sample_history = cleaned_for_forecast[
    (cleaned_for_forecast["Store ID"] == sample_store)
    & (cleaned_for_forecast["Product ID"] == sample_product)
].sort_values("Date")

sample_forecast = recursive_forecast_one_series(
    sample_history,
    horizon=FORECAST_HORIZON
)

print("Store:", sample_store)
print("Product:", sample_product)
print("Forecast origin:", sample_history["Date"].max())
display(sample_forecast)


Store: S001
Product: P0001
Forecast origin: 2024-01-30 00:00:00


,Date,Store ID,Product ID,Forecast_Horizon,Predicted_Demand
0,2024-01-31,S001,P0001,1,111.727005
1,2024-02-01,S001,P0001,2,108.130600
2,2024-02-02,S001,P0001,3,104.869003
3,2024-02-03,S001,P0001,4,106.640495
4,2024-02-04,S001,P0001,5,106.151558
5,2024-02-05,S001,P0001,6,105.708855
6,2024-02-06,S001,P0001,7,104.378746


In [12]:

# Ensure dates have consistent types
cleaned_df["Date"] = pd.to_datetime(cleaned_df["Date"])
model_df["Date"] = pd.to_datetime(model_df["Date"])

# Select the same store-product series independently in each dataframe
cleaned_series = cleaned_df[
    (cleaned_df["Store ID"] == sample_store)
    & (cleaned_df["Product ID"] == sample_product)
].sort_values("Date").reset_index(drop=True)

engineered_series = model_df[
    (model_df["Store ID"] == sample_store)
    & (model_df["Product ID"] == sample_product)
].sort_values("Date").reset_index(drop=True)

print("Store:", sample_store)
print("Product:", sample_product)
print("Cleaned series rows:", len(cleaned_series))
print("Engineered series rows:", len(engineered_series))

print("\nLatest dates:")
print("Cleaned:", cleaned_series["Date"].max())
print("Engineered:", engineered_series["Date"].max())

print("\nLatest three engineered rows:")
features_to_check = [
    "Date", "Demand",
    "Lag_1", "Lag_7", "Lag_14", "Lag_30",
    "RollingMean_7", "RollingMean_14", "RollingMean_30",
    "RollingStd_7", "RollingStd_14", "RollingStd_30",
    "Inventory_Lag_1", "Inventory_Lag_7",
    "UnitsSold_Lag_1", "UnitsSold_Lag_7", "UnitsSold_Lag_14",
    "UnitsOrdered_Lag_1", "UnitsOrdered_Lag_7",
    "UnitsOrdered_Lag_14",
    "Price_Lag_1", "Price_Change", "Price_Change_Pct",
    "CompetitorPrice_Lag_1", "CompetitorPrice_Change",
    "Inventory_Demand_Ratio", "Inventory_Coverage_Approx",
]

display(engineered_series[features_to_check].tail(3))


Store: S001
Product: P0001
Cleaned series rows: 760
Engineered series rows: 730

Latest dates:
Cleaned: 2024-01-30 00:00:00
Engineered: 2024-01-30 00:00:00

Latest three engineered rows:


,Date,Demand,Lag_1,Lag_7,Lag_14,Lag_30,RollingMean_7,RollingMean_14,RollingMean_30,RollingStd_7,...,UnitsOrdered_Lag_1,UnitsOrdered_Lag_7,UnitsOrdered_Lag_14,Price_Lag_1,Price_Change,Price_Change_Pct,CompetitorPrice_Lag_1,CompetitorPrice_Change,Inventory_Demand_Ratio,Inventory_Coverage_Approx
727,2024-01-28,139,99.0,109.0,74.0,90.0,114.428571,101.785714,85.033333,51.201748,...,0.0,0.0,0.0,72.78,9.47,0.130118,78.04,15.11,3.403670,3.242197
728,2024-01-29,112,139.0,218.0,114.0,54.0,118.714286,106.428571,86.666667,51.922103,...,229.0,368.0,265.0,82.25,-2.37,-0.028815,93.15,-17.56,1.188073,2.181709
729,2024-01-30,111,112.0,68.0,31.0,60.0,103.571429,106.285714,88.600000,28.159410,...,0.0,0.0,0.0,79.88,-8.32,-0.104156,75.59,-8.93,5.632353,3.697931


In [13]:

latest_cleaned = cleaned_series.iloc[-1]
latest_engineered = engineered_series.iloc[-1]

raw_columns = [
    "Demand",
    "Inventory Level",
    "Price",
    "Discount",
    "Competitor Pricing",
    "Promotion",
    "Weather Condition",
    "Seasonality",
    "Epidemic",
]

comparison = []

for col in raw_columns:
    comparison.append({
        "Column": col,
        "Cleaned_Value": latest_cleaned[col],
        "Model_Dataset_Value": latest_engineered[col],
        "Matches": (
            latest_cleaned[col] == latest_engineered[col]
            or (
                pd.isna(latest_cleaned[col])
                and pd.isna(latest_engineered[col])
            )
        ),
    })

display(pd.DataFrame(comparison))

print(
    "\nUnits Sold (raw dataset):",
    latest_cleaned["Units Sold"]
)
print(
    "Units Ordered (raw dataset):",
    latest_cleaned["Units Ordered"]
)


,Column,Cleaned_Value,Model_Dataset_Value,Matches
0,Demand,111,111,True
1,Inventory Level,383,383,True
2,Price,71.56,71.56,True
3,Discount,10,10,True
4,Competitor Pricing,66.66,66.66,True
5,Promotion,0,0,True
6,Weather Condition,Snowy,Snowy,True
7,Seasonality,Winter,Winter,True
8,Epidemic,0,0,True



Units Sold (raw dataset): 68
Units Ordered (raw dataset): 0


In [14]:

# Features that are not safe as future inputs without additional planning data.
# These values describe the target day or depend on contemporaneous outcomes.

UNSAFE_FUTURE_FEATURES = [
    "Inventory Level",           # Future opening stock is not yet known
    "Inventory_Demand_Ratio",    # Uses same-day Units Sold
    "Inventory_Coverage_Approx", # Depends on future inventory assumptions
    "Price",                     # Future price not guaranteed known
    "Discount",                  # Future discount not guaranteed known
    "Competitor Pricing",        # Future competitor price not known
    "Weather Condition",         # Future actual weather is not known
    "Promotion",                 # Use only if future promotions are planned
    "Epidemic",                  # Future status not guaranteed known
    "Price_Change",
    "Price_Change_Pct",
    "CompetitorPrice_Change",
    "Price_Competitor_Difference",
]

SAFE_FEATURES = [
    col for col in FEATURE_COLUMNS
    if col not in UNSAFE_FUTURE_FEATURES
]

print("Original feature count:", len(FEATURE_COLUMNS))
print("Features removed:", len(FEATURE_COLUMNS) - len(SAFE_FEATURES))
print("Safe feature count:", len(SAFE_FEATURES))

print("\nRemoved features:")
print(UNSAFE_FUTURE_FEATURES)

print("\nRetained features:")
print(SAFE_FEATURES)

assert "Demand" not in SAFE_FEATURES
assert "Units Sold" not in SAFE_FEATURES
assert "Inventory_Demand_Ratio" not in SAFE_FEATURES


Original feature count: 53
Features removed: 13
Safe feature count: 40

Removed features:
['Inventory Level', 'Inventory_Demand_Ratio', 'Inventory_Coverage_Approx', 'Price', 'Discount', 'Competitor Pricing', 'Weather Condition', 'Promotion', 'Epidemic', 'Price_Change', 'Price_Change_Pct', 'CompetitorPrice_Change', 'Price_Competitor_Difference']

Retained features:
['Store ID', 'Product ID', 'Category', 'Region', 'Seasonality', 'Year', 'Month', 'Quarter', 'DayOfMonth', 'DayOfWeek', 'WeekOfYear', 'IsWeekend', 'DayOfWeek_Sin', 'DayOfWeek_Cos', 'Month_Sin', 'Month_Cos', 'Lag_1', 'Lag_7', 'Lag_14', 'Lag_30', 'RollingMean_7', 'RollingMean_14', 'RollingMean_30', 'RollingStd_7', 'RollingStd_14', 'RollingStd_30', 'RollingMin_7', 'RollingMax_7', 'RollingMin_30', 'RollingMax_30', 'Inventory_Lag_1', 'Inventory_Lag_7', 'UnitsSold_Lag_1', 'UnitsSold_Lag_7', 'UnitsSold_Lag_14', 'UnitsOrdered_Lag_1', 'UnitsOrdered_Lag_7', 'UnitsOrdered_Lag_14', 'Price_Lag_1', 'CompetitorPrice_Lag_1']


In [15]:
print("Safe features grouped by type")
print("-" * 45)

safe_categorical = [
    c for c in CATEGORICAL_FEATURES if c in SAFE_FEATURES
]
safe_numerical = [
    c for c in NUMERICAL_FEATURES if c in SAFE_FEATURES
]

print("Categorical:", safe_categorical)
print("\nNumerical:", safe_numerical)

print("\nAny missing columns?")
missing = set(SAFE_FEATURES) - set(train_df.columns)
print(missing)

assert not missing


Safe features grouped by type
---------------------------------------------
Categorical: ['Store ID', 'Product ID', 'Category', 'Region', 'Seasonality']

Numerical: ['Year', 'Month', 'Quarter', 'DayOfMonth', 'DayOfWeek', 'WeekOfYear', 'IsWeekend', 'DayOfWeek_Sin', 'DayOfWeek_Cos', 'Month_Sin', 'Month_Cos', 'Lag_1', 'Lag_7', 'Lag_14', 'Lag_30', 'RollingMean_7', 'RollingMean_14', 'RollingMean_30', 'RollingStd_7', 'RollingStd_14', 'RollingStd_30', 'RollingMin_7', 'RollingMax_7', 'RollingMin_30', 'RollingMax_30', 'Inventory_Lag_1', 'Inventory_Lag_7', 'UnitsSold_Lag_1', 'UnitsSold_Lag_7', 'UnitsSold_Lag_14', 'UnitsOrdered_Lag_1', 'UnitsOrdered_Lag_7', 'UnitsOrdered_Lag_14', 'Price_Lag_1', 'CompetitorPrice_Lag_1']

Any missing columns?
set()


In [16]:

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# Define the safe feature types
safe_categorical = [
    col for col in CATEGORICAL_FEATURES if col in SAFE_FEATURES
]
safe_numerical = [
    col for col in NUMERICAL_FEATURES if col in SAFE_FEATURES
]

# Separate predictors and target
X_train = train_df[SAFE_FEATURES].copy()
y_train = train_df[TARGET_COL].copy()

X_val = validation_df[SAFE_FEATURES].copy()
y_val = validation_df[TARGET_COL].copy()

# Preprocess categorical and numerical features
safe_preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            OneHotEncoder(handle_unknown="ignore"),
            safe_categorical
        ),
        ("numerical", "passthrough", safe_numerical),
    ],
    remainder="drop"
)

# Use the best parameters from the previous tuning run
safe_xgb = XGBRegressor(
    n_estimators=500,
    max_depth=7,
    learning_rate=0.08,
    objective="reg:squarederror",
    random_state=RANDOM_SEED,
    n_jobs=-1
)

safe_model = Pipeline([
    ("preprocessor", safe_preprocessor),
    ("model", safe_xgb)
])

# Fit on training data only
safe_model.fit(X_train, y_train)

# Validation predictions
val_predictions = np.maximum(0, safe_model.predict(X_val))

def calculate_metrics(y_true, y_pred):
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    denominator = np.abs(y_true) + np.abs(y_pred)
    valid = denominator > 0

    smape = (
        100 * np.mean(
            2 * np.abs(y_true[valid] - y_pred[valid])
            / denominator[valid]
        )
        if valid.any() else np.nan
    )

    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
        "sMAPE": smape,
        "R2": r2_score(y_true, y_pred),
    }

safe_validation_metrics = calculate_metrics(y_val, val_predictions)

print("LEAKAGE-REDUCED MODEL — VALIDATION RESULTS")
print("-" * 50)
for metric, value in safe_validation_metrics.items():
    print(f"{metric}: {value:.4f}")


LEAKAGE-REDUCED MODEL — VALIDATION RESULTS
--------------------------------------------------
MAE: 35.2706
RMSE: 46.1650
sMAPE: 32.9720
R2: 0.0929


In [17]:

safe_model_path = MODELS_DIR / "xgboost_7day_safe.pkl"
safe_metadata_path = MODELS_DIR / "xgboost_7day_safe_metadata.json"

joblib.dump(safe_model, safe_model_path)

safe_metadata = {
    "model": "XGBoost",
    "purpose": "Leakage-reduced 7-day recursive forecasting",
    "target": TARGET_COL,
    "forecast_horizon": FORECAST_HORIZON,
    "features": SAFE_FEATURES,
    "categorical_features": safe_categorical,
    "numerical_features": safe_numerical,
    "best_parameters": {
        "n_estimators": 500,
        "max_depth": 7,
        "learning_rate": 0.08
    },
    "train_rows": int(len(X_train)),
    "validation_rows": int(len(X_val)),
    "validation_metrics": {
        key: float(value)
        for key, value in safe_validation_metrics.items()
    },
    "future_feature_assumption": (
        "Only retained features are used; future demand lags "
        "must be updated recursively from model predictions."
    )
}

with open(safe_metadata_path, "w", encoding="utf-8") as f:
    json.dump(safe_metadata, f, indent=2)

print("Saved model:", safe_model_path)
print("Saved metadata:", safe_metadata_path)


Saved model: D:\AI-Retail-Demand-Forecasting\models\xgboost_7day_safe.pkl
Saved metadata: D:\AI-Retail-Demand-Forecasting\models\xgboost_7day_safe_metadata.json


In [18]:

# Reload the newly saved model
safe_model = joblib.load(MODELS_DIR / "xgboost_7day_safe.pkl")

# Test one validation row
test_row = X_val.iloc[[0]]
test_prediction = safe_model.predict(test_row)[0]

print("Reload successful.")
print("Validation row date:", validation_df.iloc[0]["Date"])
print("Actual demand:", y_val.iloc[0])
print("Predicted demand:", round(float(test_prediction), 2))
print("Input feature count:", test_row.shape[1])


Reload successful.
Validation row date: 2023-06-25
Actual demand: 122
Predicted demand: 120.31
Input feature count: 40


In [19]:

SAFE_FEATURES = safe_metadata["features"]

def forecast_seven_days(history, model=safe_model, horizon=7):
    history = history.sort_values("Date").copy().reset_index(drop=True)

    if len(history) < 30:
        raise ValueError("At least 30 days of history are required.")

    if history["Date"].duplicated().any():
        raise ValueError("Duplicate dates found.")

    if history["Date"].diff().dropna().dt.days.ne(1).any():
        raise ValueError("History must have continuous daily dates.")

    history_dates = pd.to_datetime(history["Date"])
    origin_date = history_dates.max()

    demand_history = history["Demand"].astype(float).tolist()
    last = history.iloc[-1]

    output = []

    for step in range(1, horizon + 1):
        date = origin_date + pd.Timedelta(days=step)

        # Derive seasonality from the future calendar date
        month = date.month
        if month in [12, 1, 2]:
            seasonality = "Winter"
        elif month in [3, 4, 5]:
            seasonality = "Spring"
        elif month in [6, 7, 8]:
            seasonality = "Summer"
        else:
            seasonality = "Autumn"

        row = {
            "Store ID": last["Store ID"],
            "Product ID": last["Product ID"],
            "Category": last["Category"],
            "Region": last["Region"],
            "Seasonality": seasonality,
            "Year": date.year,
            "Month": date.month,
            "Quarter": date.quarter,
            "DayOfMonth": date.day,
            "DayOfWeek": date.dayofweek,
            "WeekOfYear": int(date.isocalendar().week),
            "IsWeekend": int(date.dayofweek >= 5),
            "DayOfWeek_Sin": np.sin(2 * np.pi * date.dayofweek / 7),
            "DayOfWeek_Cos": np.cos(2 * np.pi * date.dayofweek / 7),
            "Month_Sin": np.sin(2 * np.pi * date.month / 12),
            "Month_Cos": np.cos(2 * np.pi * date.month / 12),
        }

        # Recursive target-history features
        for lag in [1, 7, 14, 30]:
            row[f"Lag_{lag}"] = demand_history[-lag]

        for window in [7, 14, 30]:
            values = np.asarray(demand_history[-window:], dtype=float)
            row[f"RollingMean_{window}"] = values.mean()
            row[f"RollingStd_{window}"] = values.std(ddof=1)

        row["RollingMin_7"] = min(demand_history[-7:])
        row["RollingMax_7"] = max(demand_history[-7:])
        row["RollingMin_30"] = min(demand_history[-30:])
        row["RollingMax_30"] = max(demand_history[-30:])

        # Historical operational lag features: last known values held fixed
        for feature, source, lag in [
            ("Inventory_Lag_1", "Inventory Level", 1),
            ("Inventory_Lag_7", "Inventory Level", 7),
            ("UnitsSold_Lag_1", "Units Sold", 1),
            ("UnitsSold_Lag_7", "Units Sold", 7),
            ("UnitsSold_Lag_14", "Units Sold", 14),
            ("UnitsOrdered_Lag_1", "Units Ordered", 1),
            ("UnitsOrdered_Lag_7", "Units Ordered", 7),
            ("UnitsOrdered_Lag_14", "Units Ordered", 14),
        ]:
            row[feature] = float(history[source].iloc[-lag])

        row["Price_Lag_1"] = float(last["Price"])
        row["CompetitorPrice_Lag_1"] = float(last["Competitor Pricing"])

        X_future = pd.DataFrame([row])[SAFE_FEATURES]
        prediction = max(0.0, float(model.predict(X_future)[0]))

        output.append({
            "Date": date,
            "Store ID": last["Store ID"],
            "Product ID": last["Product ID"],
            "Forecast_Horizon": step,
            "Predicted_Demand": prediction,
        })

        # Do not use future actual demand here
        demand_history.append(prediction)

    return pd.DataFrame(output)


In [20]:

cleaned_df["Date"] = pd.to_datetime(cleaned_df["Date"])

sample_store = "S001"
sample_product = "P0001"

sample_history = cleaned_df[
    (cleaned_df["Store ID"] == sample_store)
    & (cleaned_df["Product ID"] == sample_product)
].sort_values("Date")

sample_predictions = forecast_seven_days(
    sample_history,
    horizon=FORECAST_HORIZON
)

display(sample_predictions)
assert len(sample_predictions) == 7
assert sample_predictions["Date"].is_monotonic_increasing
assert sample_predictions["Predicted_Demand"].notna().all()

print("Seven-day recursive forecast passed the smoke test.")


,Date,Store ID,Product ID,Forecast_Horizon,Predicted_Demand
0,2024-01-31,S001,P0001,1,57.538372
1,2024-02-01,S001,P0001,2,75.274651
2,2024-02-02,S001,P0001,3,88.529747
3,2024-02-03,S001,P0001,4,89.552864
4,2024-02-04,S001,P0001,5,97.679863
5,2024-02-05,S001,P0001,6,88.685341
6,2024-02-06,S001,P0001,7,73.970139


Seven-day recursive forecast passed the smoke test.


In [21]:

# Check which season labels the dataset uses for each month
cleaned_df["Date"] = pd.to_datetime(cleaned_df["Date"])

season_check = pd.crosstab(
    cleaned_df["Date"].dt.month,
    cleaned_df["Seasonality"],
    normalize="index"
).round(3)

display(season_check)


Seasonality,Autumn,Spring,Summer,Winter
Date,,,,
1,0.0,0.0,0.0,1.0
2,0.0,0.0,0.0,1.0
3,0.0,1.0,0.0,0.0
4,0.0,1.0,0.0,0.0
5,0.0,1.0,0.0,0.0
6,0.0,0.0,1.0,0.0
7,0.0,0.0,1.0,0.0
8,0.0,0.0,1.0,0.0
9,1.0,0.0,0.0,0.0


In [22]:

# Evaluate the leakage-reduced model on the validation set
val_predictions = np.maximum(0, safe_model.predict(X_val))

safe_validation_metrics = calculate_metrics(
    y_val,
    val_predictions
)

print("Safe model validation metrics:")
for metric, value in safe_validation_metrics.items():
    print(f"{metric}: {value:.4f}")


Safe model validation metrics:
MAE: 35.2706
RMSE: 46.1650
sMAPE: 32.9720
R2: 0.0929


In [23]:

correlations = (
    train_df[SAFE_FEATURES + [TARGET_COL]]
    .corr(numeric_only=True)[TARGET_COL]
    .drop(TARGET_COL)
    .sort_values(key=lambda s: s.abs(), ascending=False)
)

display(correlations.to_frame("Correlation_with_Demand").head(20))


,Correlation_with_Demand
RollingMean_7,0.501780
RollingMean_14,0.484177
RollingMax_7,0.450885
RollingMean_30,0.399253
RollingMin_7,0.369794
Lag_1,0.367210
RollingStd_14,0.325463
RollingMax_30,0.324938
UnitsSold_Lag_1,0.318166
RollingStd_7,0.314916


In [24]:

# Compare actual and predicted demand on the validation set
val_predictions = np.maximum(0, safe_model.predict(X_val))

validation_diagnostic = pd.DataFrame({
    "Actual_Demand": y_val.to_numpy(),
    "Predicted_Demand": val_predictions
})

validation_diagnostic["Error"] = (
    validation_diagnostic["Predicted_Demand"]
    - validation_diagnostic["Actual_Demand"]
)

validation_diagnostic["Absolute_Error"] = (
    validation_diagnostic["Error"].abs()
)

print("Actual demand summary:")
display(validation_diagnostic["Actual_Demand"].describe())

print("Predicted demand summary:")
display(validation_diagnostic["Predicted_Demand"].describe())

print("Mean prediction bias:", round(validation_diagnostic["Error"].mean(), 3))
print("Median absolute error:", round(validation_diagnostic["Absolute_Error"].median(), 3))

print("\nLargest absolute errors:")
display(
    validation_diagnostic
    .sort_values("Absolute_Error", ascending=False)
    .head(10)
)


Actual demand summary:


count    11000.000000
mean       112.674182
std         48.472427
min          4.000000
25%         79.000000
50%        107.000000
75%        141.000000
max        430.000000
Name: Actual_Demand, dtype: float64

Predicted demand summary:


count    11000.000000
mean       108.727440
std         25.369329
min         21.046471
25%         91.802198
50%        107.151245
75%        122.740276
max        221.198013
Name: Predicted_Demand, dtype: float64

Mean prediction bias: -3.947
Median absolute error: 28.056

Largest absolute errors:


,Actual_Demand,Predicted_Demand,Error,Absolute_Error
387,430,115.227005,-314.772995,314.772995
1213,328,88.735733,-239.264267,239.264267
2454,326,93.728844,-232.271156,232.271156
887,323,100.332802,-222.667198,222.667198
769,293,72.022690,-220.977310,220.977310
1183,298,80.726204,-217.273796,217.273796
6261,370,153.499146,-216.500854,216.500854
2163,327,114.873917,-212.126083,212.126083
2123,292,85.910095,-206.089905,206.089905
954,287,83.003220,-203.996780,203.996780


In [25]:

date_diagnostic = validation_df[["Date"]].copy()
date_diagnostic["Actual_Demand"] = y_val.to_numpy()
date_diagnostic["Predicted_Demand"] = val_predictions

date_diagnostic["Date"] = pd.to_datetime(date_diagnostic["Date"])
date_diagnostic["Absolute_Error"] = (
    date_diagnostic["Actual_Demand"]
    - date_diagnostic["Predicted_Demand"]
).abs()

date_diagnostic["Month"] = date_diagnostic["Date"].dt.to_period("M")

monthly_errors = (
    date_diagnostic.groupby("Month")
    .agg(
        MAE=("Absolute_Error", "mean"),
        Actual_Mean=("Actual_Demand", "mean"),
        Predicted_Mean=("Predicted_Demand", "mean"),
        Rows=("Actual_Demand", "size")
    )
    .round(2)
)

display(monthly_errors)


,MAE,Actual_Mean,Predicted_Mean,Rows
Month,,,,
2023-06,37.79,118.58,106.449997,600
2023-07,39.61,119.34,98.570000,3100
2023-08,34.21,120.11,119.199997,3100
2023-09,29.39,106.78,108.830002,3000
2023-10,40.23,88.04,108.809998,1200


In [26]:

# Predict on both training and validation sets
train_predictions = np.maximum(0, safe_model.predict(X_train))
val_predictions = np.maximum(0, safe_model.predict(X_val))

print("TRAINING METRICS")
for metric, value in calculate_metrics(y_train, train_predictions).items():
    print(f"{metric}: {value:.4f}")

print("\nVALIDATION METRICS")
for metric, value in calculate_metrics(y_val, val_predictions).items():
    print(f"{metric}: {value:.4f}")

print("\nTARGET DISTRIBUTION")
display(
    pd.DataFrame({
        "Train": y_train.describe(),
        "Validation": y_val.describe()
    }).round(2)
)


TRAINING METRICS
MAE: 19.4288
RMSE: 25.4360
sMAPE: 21.3795
R2: 0.7081

VALIDATION METRICS
MAE: 35.2706
RMSE: 46.1650
sMAPE: 32.9720
R2: 0.0929

TARGET DISTRIBUTION


,Train,Validation
count,51000.00,11000.00
mean,103.11,112.67
std,47.08,48.47
min,4.00,4.00
25%,70.00,79.00
50%,99.00,107.00
75%,132.00,141.00
max,378.00,430.00


In [27]:

preprocessor = safe_model.named_steps["preprocessor"]
xgb_estimator = safe_model.named_steps["model"]

feature_names = preprocessor.get_feature_names_out()
importances = xgb_estimator.feature_importances_

importance_df = (
    pd.DataFrame({
        "Feature": feature_names,
        "Importance": importances
    })
    .sort_values("Importance", ascending=False)
    .reset_index(drop=True)
)

display(importance_df.head(20))


,Feature,Importance
0,numerical__RollingMean_7,0.167537
1,categorical__Category_Furniture,0.124205
2,categorical__Category_Groceries,0.064286
3,categorical__Seasonality_Summer,0.047056
4,numerical__Month_Cos,0.027266
5,categorical__Category_Toys,0.022047
6,categorical__Category_Clothing,0.021785
7,categorical__Category_Electronics,0.020736
8,numerical__WeekOfYear,0.018589
9,numerical__Year,0.017917


In [28]:

# Compare the safe XGBoost model against simple baselines.
# All baselines use features available in the validation rows.

baseline_predictions = {
    "Safe XGBoost": np.maximum(0, safe_model.predict(X_val)),
    "Seasonal Naive (Lag_7)": np.maximum(
        0, validation_df["Lag_7"].to_numpy()
    ),
    "7-Day Moving Average": np.maximum(
        0, validation_df["RollingMean_7"].to_numpy()
    ),
    "Previous Day (Lag_1)": np.maximum(
        0, validation_df["Lag_1"].to_numpy()
    ),
}

baseline_results = []

for name, predictions in baseline_predictions.items():
    metrics = calculate_metrics(y_val, predictions)
    baseline_results.append({
        "Model": name,
        **metrics
    })

baseline_comparison = (
    pd.DataFrame(baseline_results)
    .sort_values("MAE")
    .reset_index(drop=True)
)

display(baseline_comparison.round(4))


,Model,MAE,RMSE,sMAPE,R2
0,7-Day Moving Average,34.0632,43.9804,31.4570,0.1767
1,Safe XGBoost,35.2706,46.1650,32.9720,0.0929
2,Previous Day (Lag_1),44.4992,57.7874,40.7519,-0.4214
3,Seasonal Naive (Lag_7),45.8952,59.1961,41.7948,-0.4915
